# Fundamental Agent Prototype

This notebook is intentionally thin. All prompts, schemas, and reusable functions live in `src/openclam/agents/fundamental/fundamental_agent.py`, and the notebook only imports and calls those functions.

## 1. Environment Setup

Load the repo root, make `src/` importable, and load local environment variables from `.env` when available.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / "pyproject.toml").exists() else CURRENT_DIR.parent

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

loaded_env = load_dotenv(REPO_ROOT / ".env", override=True)
print(f"Repo root: {REPO_ROOT}")
print(f"Loaded .env: {loaded_env}")

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if userdata is not None:
    openai_key = userdata.get("OPENAI_API_KEY")
    fmp_key = userdata.get("FMP_API_KEY")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
    if fmp_key:
        os.environ["FMP_API_KEY"] = fmp_key

print(f"OPENAI_API_KEY loaded: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"FMP_API_KEY loaded: {bool(os.getenv('FMP_API_KEY'))}")


Repo root: C:\Users\22840\OneDrive\Desktop\genai_project\OpenClam_Multi-Agent_Investment_Advisory
Loaded .env: True
OPENAI_API_KEY loaded: True
FMP_API_KEY loaded: True


## 2. Import Fundamental Module

Reload the module during local iteration so notebook state does not hold stale definitions.

In [2]:
import importlib
import openclam.agents.fundamental.fundamental_agent as fundamental_agent

fundamental_agent = importlib.reload(fundamental_agent)

analyze_fundamental_input = fundamental_agent.analyze_fundamental_input
create_fundamental_agent = fundamental_agent.create_fundamental_agent
get_yfinance_fundamental_input = fundamental_agent.get_yfinance_fundamental_input
run_fundamental_analysis = fundamental_agent.run_fundamental_analysis
run_fundamental_workflow = fundamental_agent.run_fundamental_workflow

print(f"Loaded module from: {fundamental_agent.__file__}")


Loaded module from: C:\Users\22840\OneDrive\Desktop\genai_project\OpenClam_Multi-Agent_Investment_Advisory\src\openclam\agents\fundamental\fundamental_agent.py


## 3. Configure the Run

Set the ticker, transcript window, and model configuration used by the module helpers.

In [3]:
TICKER = "AAPL"
TRANSCRIPT_YEAR = 2025
TRANSCRIPT_QUARTER = 4
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
TEMPERATURE = 0.0


## 4. Create the Agent Client

This step builds the reusable agent object from the Python module.

In [4]:
agent = create_fundamental_agent(
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
)


## 5. Build Structured Fundamental Input

Generate the reusable structured input payload from market data and optional transcript context.

In [5]:
fundamental_input = get_yfinance_fundamental_input(
    TICKER,
    transcript_year=TRANSCRIPT_YEAR,
    transcript_quarter=TRANSCRIPT_QUARTER,
    require_transcript=True,
)

print(f"Transcript snippets loaded: {len(fundamental_input.earnings_call_snippets)}")
print(f"Guidance text available: {bool(fundamental_input.guidance_text)}")
print("===== AUTO-GENERATED INPUT =====")
print(fundamental_input.model_dump_json(indent=2))


Transcript snippets loaded: 20
Guidance text available: True
===== AUTO-GENERATED INPUT =====
{
  "ticker": "AAPL",
  "company_name": "Apple Inc.",
  "quarter": "2025-12-31",
  "earnings_summary": "\n    Company: Apple Inc.\n    Ticker: AAPL\n    Latest valid financial quarter: 2025-12-31\n\n    Important: current vs prior values below are quarter-over-quarter comparisons, NOT year-over-year.\n\n    Revenue QoQ: current=143756000000.0, previous_quarter=102466000000.0\n    Gross Profit QoQ: current=69231000000.0, previous_quarter=48341000000.0\n    Operating Income QoQ: current=50852000000.0, previous_quarter=32427000000.0\n    Net Income QoQ: current=42097000000.0, previous_quarter=27466000000.0\n    Operating Cash Flow QoQ: current=53925000000.0, previous_quarter=29728000000.0\n    Free Cash Flow QoQ: current=51552000000.0, previous_quarter=26486000000.0\n\n    Market Cap: 4220926033920.0\n    Trailing P/E: 34.7503\n    Forward P/E: 30.026237\n    Profit Margin: 0.27152002\n    Revenu

## 6. Run the Fundamental Analysis

Pass the structured input into the module's analysis helper and print the final JSON output.

In [6]:
result = analyze_fundamental_input(
    fundamental_input,
    agent=agent,
)

print("\n===== FUNDAMENTAL AGENT OUTPUT =====")
print(result.model_dump_json(indent=2))



===== FUNDAMENTAL AGENT OUTPUT =====
{
  "ticker": "AAPL",
  "stance": "Bullish",
  "core_judgment": "Strong revenue growth and profitability with positive management outlook.",
  "positive_signals": [
    "Revenue increased significantly to $143.76 billion from $102.47 billion QoQ.",
    "Net income rose to $42.1 billion from $27.5 billion QoQ.",
    "Operating cash flow reached a record $53.9 billion, indicating strong cash generation.",
    "Management expects December quarter revenue to grow by 10% to 12% YoY, indicating confidence in continued growth."
  ],
  "negative_signals": [
    "Total debt is $90.5 billion, which could pose leverage risks.",
    "Operating expenses increased by 11% YoY, which may pressure margins if not managed carefully.",
    "Tariff-related costs of approximately $1.4 billion could impact profitability."
  ],
  "key_evidence": [
    "Management reported an all-time revenue record of $416 billion for fiscal year 2025.",
    "Services revenue reached an a

## 7. Optional Single-Call Workflow

Use this when you want both the generated input payload and the final model output from one module function.

```python
workflow_result = run_fundamental_workflow(
    TICKER,
    transcript_year=TRANSCRIPT_YEAR,
    transcript_quarter=TRANSCRIPT_QUARTER,
    require_transcript=True,
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
)

print(workflow_result.model_dump_json(indent=2))
```

## 8. Evaluation: Q4 2025 Earnings Window for Magnificent Seven

This section mirrors the market technical and news macro demos. It uses the same Mag 7 Q4 2025 earnings cases, builds post-earnings event-window returns versus SPY and QQQ, and then runs the fundamental agent across the full set.

The evaluation helper also filters quarterly statement snapshots to each earnings date so the backtest does not leak later reported quarters. Because the fundamental agent currently emits one primary stance, the helper infers a short-term event stance from beat/miss, guidance, management tone, and thesis-impact fields, while preserving the agent's native stance as the long-term call.

In [7]:
mag7_q4_2025_earnings_df = fundamental_agent.mag7_q4_2025_earnings_df
build_earnings_price_eval = fundamental_agent.build_earnings_price_eval
run_agent_event_window_eval = fundamental_agent.run_agent_event_window_eval
summarize_eval_results = fundamental_agent.summarize_eval_results

PRE_TRADING_DAYS = 7
POST_TRADING_DAYS = 7
LONG_POST_TRADING_DAYS = 30

earnings_df = mag7_q4_2025_earnings_df()
earnings_df


,ticker,company,earnings_date
0,TSLA,Tesla,2026-01-28
1,META,Meta Platforms,2026-01-28
2,AAPL,Apple,2026-01-29
3,MSFT,Microsoft,2026-01-28
4,GOOGL,Alphabet,2026-02-03
5,AMZN,Amazon,2026-02-05
6,NVDA,Nvidia,2026-02-25


In [8]:
summary_df, paths_df = build_earnings_price_eval(
    earnings_df=earnings_df,
    pre_trading_days=PRE_TRADING_DAYS,
    post_trading_days=POST_TRADING_DAYS,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    benchmarks=("SPY", "QQQ"),
)

summary_df[[
    "ticker",
    "company",
    "earnings_date",
    "post_7d_return",
    "abnormal_vs_qqq",
    "post_30d_return",
    "abnormal_30d_vs_qqq",
]]


,ticker,company,earnings_date,post_7d_return,abnormal_vs_qqq,post_30d_return,abnormal_30d_vs_qqq
0,TSLA,Tesla,2026-01-28,-0.047165,-0.009943,-0.084481,-0.027692
1,META,Meta Platforms,2026-01-28,-0.010871,0.026351,-0.045684,0.011106
2,AAPL,Apple,2026-01-29,0.064260,0.088265,-0.030687,0.026046
3,MSFT,Microsoft,2026-01-28,-0.167120,-0.129898,-0.163721,-0.106932
4,GOOGL,Alphabet,2026-02-03,-0.090401,-0.064643,-0.093619,-0.058552
5,AMZN,Amazon,2026-02-05,-0.096726,-0.103878,-0.077776,-0.052702
6,NVDA,Nvidia,2026-02-25,-0.090714,-0.063260,-0.059522,-0.050245


In [9]:
agent_eval, agent_reports = run_agent_event_window_eval(
    summary_df,
    transcript_year=2025,
    transcript_quarter=4,
    require_transcript=False,
    agent=agent,
    neutral_band=0.02,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
)

agent_eval


,ticker,company,earnings_date,post_7d_return,abnormal_vs_qqq,realized_direction_vs_qqq,post_30d_return,abnormal_30d_vs_qqq,realized_30d_direction_vs_qqq,news_context_ready,...,agent_thesis_impact,agent_beat_or_miss,confidence_rationale,stance_rationale,short_direction_match,short_direction_match_reason,long_direction_match,long_direction_match_reason,direction_match,direction_match_reason
0,TSLA,Tesla,2026-01-28,-0.047165,-0.009943,down,-0.084481,-0.027692,down,True,...,Weakened,Unclear,While management expresses confidence in futur...,Fundamentals are weakening with declining reve...,None,prediction or realization missing,True,correctly predicted down move (-2.77%),None,prediction or realization missing
1,META,Meta Platforms,2026-01-28,-0.010871,0.026351,up,-0.045684,0.011106,up,True,...,Strengthened,Unclear,The substantial increase in revenue and net in...,Strong revenue growth and positive outlook dri...,True,correctly predicted up move (2.64%),None,moved within neutral band (2.0%),True,correctly predicted up move (2.64%)
2,AAPL,Apple,2026-01-29,0.064260,0.088265,up,-0.030687,0.026046,up,True,...,Strengthened,Unclear,The substantial QoQ improvements in revenue an...,Strong revenue growth and positive management ...,True,correctly predicted up move (8.83%),True,correctly predicted up move (2.60%),True,correctly predicted up move (8.83%)
3,MSFT,Microsoft,2026-01-28,-0.167120,-0.129898,down,-0.163721,-0.106932,down,True,...,Strengthened,Beat,The substantial growth in key segments and pos...,Strong performance with significant growth in ...,False,predicted up but realized down (-12.99%),False,predicted up but realized down (-10.69%),False,predicted up but realized down (-12.99%)
4,GOOGL,Alphabet,2026-02-03,-0.090401,-0.064643,down,-0.093619,-0.058552,down,True,...,Strengthened,Unclear,The combination of strong financial performanc...,Strong revenue growth driven by AI investments...,False,predicted up but realized down (-6.46%),False,predicted up but realized down (-5.86%),False,predicted up but realized down (-6.46%)
5,AMZN,Amazon,2026-02-05,-0.096726,-0.103878,down,-0.077776,-0.052702,down,True,...,Strengthened,Unclear,The management's optimistic tone and clear gro...,Strong revenue growth and operational improvem...,False,predicted up but realized down (-10.39%),False,predicted up but realized down (-5.27%),False,predicted up but realized down (-10.39%)
6,NVDA,Nvidia,2026-02-25,-0.090714,-0.063260,down,-0.059522,-0.050245,down,True,...,Strengthened,Beat,Management's positive outlook and strong perfo...,NVIDIA Corporation is experiencing strong reve...,False,predicted up but realized down (-6.33%),False,predicted up but realized down (-5.02%),False,predicted up but realized down (-6.33%)


In [10]:
display_cols = [
    "ticker",
    "company",
    "earnings_date",
    "post_7d_return",
    "abnormal_vs_qqq",
    "realized_direction_vs_qqq",
    "post_30d_return",
    "abnormal_30d_vs_qqq",
    "realized_30d_direction_vs_qqq",
    "agent_short_term_stance",
    "agent_long_term_stance",
    "agent_guidance_change",
    "agent_management_tone",
    "agent_thesis_impact",
    "agent_beat_or_miss",
    "agent_confidence",
    "short_direction_match",
    "short_direction_match_reason",
    "long_direction_match",
    "long_direction_match_reason",
]

available_display_cols = [col for col in display_cols if col in agent_eval.columns]
agent_eval[available_display_cols]


,ticker,company,earnings_date,post_7d_return,abnormal_vs_qqq,realized_direction_vs_qqq,post_30d_return,abnormal_30d_vs_qqq,realized_30d_direction_vs_qqq,agent_short_term_stance,agent_long_term_stance,agent_guidance_change,agent_management_tone,agent_thesis_impact,agent_beat_or_miss,agent_confidence,short_direction_match,short_direction_match_reason,long_direction_match,long_direction_match_reason
0,TSLA,Tesla,2026-01-28,-0.047165,-0.009943,down,-0.084481,-0.027692,down,Neutral,Bearish,Unclear,Optimistic,Weakened,Unclear,0.5,None,prediction or realization missing,True,correctly predicted down move (-2.77%)
1,META,Meta Platforms,2026-01-28,-0.010871,0.026351,up,-0.045684,0.011106,up,Bullish,Bullish,Maintain,Optimistic,Strengthened,Unclear,0.8,True,correctly predicted up move (2.64%),None,moved within neutral band (2.0%)
2,AAPL,Apple,2026-01-29,0.064260,0.088265,up,-0.030687,0.026046,up,Bullish,Bullish,Maintain,Positive,Strengthened,Unclear,0.8,True,correctly predicted up move (8.83%),True,correctly predicted up move (2.60%)
3,MSFT,Microsoft,2026-01-28,-0.167120,-0.129898,down,-0.163721,-0.106932,down,Bullish,Bullish,Maintain,Optimistic,Strengthened,Beat,0.85,False,predicted up but realized down (-12.99%),False,predicted up but realized down (-10.69%)
4,GOOGL,Alphabet,2026-02-03,-0.090401,-0.064643,down,-0.093619,-0.058552,down,Bullish,Bullish,Maintain,Confident and optimistic about future growth a...,Strengthened,Unclear,0.85,False,predicted up but realized down (-6.46%),False,predicted up but realized down (-5.86%)
5,AMZN,Amazon,2026-02-05,-0.096726,-0.103878,down,-0.077776,-0.052702,down,Bullish,Bullish,Maintain,Optimistic,Strengthened,Unclear,0.8,False,predicted up but realized down (-10.39%),False,predicted up but realized down (-5.27%)
6,NVDA,Nvidia,2026-02-25,-0.090714,-0.063260,down,-0.059522,-0.050245,down,Bullish,Bullish,Maintain,Positive,Strengthened,Beat,0.8,False,predicted up but realized down (-6.33%),False,predicted up but realized down (-5.02%)


In [11]:
eval_summary = summarize_eval_results(agent_eval)

print("Short-Term (7-day) Analysis:")
print(f"  Evaluable cases: {eval_summary['short_evaluable_cases']}")
print(f"  Correct predictions: {eval_summary['short_matched_cases']}")
print(f"  Missed predictions: {eval_summary['short_missed_cases']}")
print(f"  Accuracy: {eval_summary['short_accuracy']:.2%}" if eval_summary['short_accuracy'] is not None else "  Accuracy: N/A")
print(f"  Neutral stance rate: {eval_summary['short_neutral_rate']:.2%}" if eval_summary['short_neutral_rate'] is not None else "  Neutral stance rate: N/A")

print("\nLong-Term (30-day) Analysis:")
print(f"  Evaluable cases: {eval_summary['long_evaluable_cases']}")
print(f"  Correct predictions: {eval_summary['long_matched_cases']}")
print(f"  Missed predictions: {eval_summary['long_missed_cases']}")
print(f"  Accuracy: {eval_summary['long_accuracy']:.2%}" if eval_summary['long_accuracy'] is not None else "  Accuracy: N/A")
print(f"  Neutral stance rate: {eval_summary['long_neutral_rate']:.2%}" if eval_summary['long_neutral_rate'] is not None else "  Neutral stance rate: N/A")

print("\nSample agent outputs:")
for ticker in list(agent_reports)[:2]:
    report = agent_reports[ticker]
    print(f"\n{ticker}")
    print(f"  Stance: {report.stance}")
    print(f"  Guidance: {report.guidance_change}")
    print(f"  Tone: {report.management_tone}")
    print(f"  Thesis impact: {report.thesis_impact}")
    print(f"  Confidence: {report.confidence:.2f}")

eval_summary


Short-Term (7-day) Analysis:
  Evaluable cases: 6
  Correct predictions: 2
  Missed predictions: 4
  Accuracy: 33.33%
  Neutral stance rate: 14.29%

Long-Term (30-day) Analysis:
  Evaluable cases: 6
  Correct predictions: 2
  Missed predictions: 4
  Accuracy: 33.33%
  Neutral stance rate: 0.00%

Sample agent outputs:

TSLA
  Stance: Bearish
  Guidance: Unclear
  Tone: Optimistic
  Thesis impact: Weakened
  Confidence: 0.50

META
  Stance: Bullish
  Guidance: Maintain
  Tone: Optimistic
  Thesis impact: Strengthened
  Confidence: 0.80


{'cases': 7,
 'short_evaluable_cases': 6,
 'short_matched_cases': 2,
 'short_missed_cases': 4,
 'short_accuracy': 0.3333333333333333,
 'short_neutral_rate': 0.14285714285714285,
 'short_not_evaluable_cases': 1,
 'short_reason_counts': {'prediction or realization missing': 1,
  'correctly predicted up move (2.64%)': 1,
  'correctly predicted up move (8.83%)': 1,
  'predicted up but realized down (-12.99%)': 1,
  'predicted up but realized down (-6.46%)': 1,
  'predicted up but realized down (-10.39%)': 1,
  'predicted up but realized down (-6.33%)': 1},
 'long_evaluable_cases': 6,
 'long_matched_cases': 2,
 'long_missed_cases': 4,
 'long_accuracy': 0.3333333333333333,
 'long_neutral_rate': 0.0,
 'long_not_evaluable_cases': 1,
 'long_reason_counts': {'correctly predicted down move (-2.77%)': 1,
  'moved within neutral band (2.0%)': 1,
  'correctly predicted up move (2.60%)': 1,
  'predicted up but realized down (-10.69%)': 1,
  'predicted up but realized down (-5.86%)': 1,
  'predicted u